# NGC 6569 full CaT + SPACE/SP_Ace + MARCS SED pipeline — v9

This notebook is the complete integrated workflow:

1. read AAT FITS spectra;
2. repair bad FITS RA/DEC header cards without modifying the originals;
3. normalize the spectra and shift them back to the stellar rest/lab frame;
4. write `aat_<fiber>.txt` files for SPACE/SP_Ace;
5. measure/check CaT Voigt fits and write a CaT master table;
6. prompt for the local HB level `imag_HB` and compute Dias & Parisi (2020) `[Fe/H]DP`;
7. run or ingest SPACE/SP_Ace full-line and alpha/metals fits;
8. run the original SEDfit_v2 MARCS-cache SED fitter, including leave-one-out rejected-band logic;
9. make the combined spectral + SED diagnostic plot for every fiber with a valid fit;
10. write the final best-practices summary table.

The SED fitting cell is based on the original `SEDfit_v2` leave-one-out MARCS code, rather than the simplified redraw that made the incorrect SED plot.

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================

from pathlib import Path
import os
from dotenv import load_dotenv
import re
import glob
import gzip
import shutil
import subprocess
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve project paths from .env, with repo-local fallbacks.
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "data/raw/fits").exists():
    candidates = [
        PROJECT_ROOT / "joanne's_code" / "SpectraReductions2026",
        Path.cwd() / "joanne's_code" / "SpectraReductions2026",
        Path.cwd().parent,
    ]
    PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "data/raw/fits").exists()), PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Main working directory for generated pipeline outputs.
BASE_DIR = PROJECT_ROOT
SPACE_DIR = Path(os.getenv("SPACE_DIR", PROJECT_ROOT / "space")).resolve()
RAW_FITS_DIR = Path(os.getenv("RAW_FITS_DIR", PROJECT_ROOT / "data/raw/fits")).resolve()
INTERIM_DATA_DIR = Path(os.getenv("INTERIM_DATA_DIR", PROJECT_ROOT / "data/interim")).resolve()
FINAL_DATA_DIR = Path(os.getenv("FINAL_DATA_DIR", PROJECT_ROOT / "data/final")).resolve()

# Output directory for this notebook
OUT_DIR = Path(os.getenv("PIPELINE_OUTPUT_DIR", PROJECT_ROOT / "outputs/NGC6569_full_CaT_SPACE_SED_pipeline_v9")).resolve()
TABLE_DIR = OUT_DIR / "tables"
PLOT_DIR = OUT_DIR / "plots"
NORMALIZED_AAT_DIR = Path(os.getenv("NORMALIZED_AAT_DIR", OUT_DIR / "aat_normalized_lab_frame")).resolve()
REPAIRED_FITS_DIR = OUT_DIR / "repaired_fits_for_pipeline"
SPACE_RUN_DIR = OUT_DIR / "space_runs"
SPACE_OUTPUT_DIR = OUT_DIR / "space_outputs"
INTEGRATED_PLOT_DIR = OUT_DIR / "integrated_CaT_SPACE_SED_diagnostics"

for d in [OUT_DIR, TABLE_DIR, PLOT_DIR, NORMALIZED_AAT_DIR, REPAIRED_FITS_DIR, SPACE_RUN_DIR, SPACE_OUTPUT_DIR, INTEGRATED_PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Input tables
PHOT_FILE = INTERIM_DATA_DIR / "CaT_summary_BDBS_XCSAO.csv"
SP_ACE_AVG_FILE = FINAL_DATA_DIR / "SP_Ace_averages_with_FeHDP_avg_std.csv"
BEST_AVERAGES_FILE = FINAL_DATA_DIR / "NGC6569_full_machine_readable_best_averages.csv"

# MARCS cache made earlier by your SED code
MARCS_DIR = Path(os.getenv("MARCS_DIR", PROJECT_ROOT / "data/raw/MARCS")).resolve()
MARCS_CACHE_FILE = MARCS_DIR / "MARCS_synthetic_ugrizy_cache.csv"
SED_OUTPUT_DIR = OUT_DIR / "MARCS_SED_PLOTS_RESTRICTED_LOO_REJECT_SPclosest_MARCS"
SED_SUMMARY_CSV = TABLE_DIR / "MARCS_batch_restricted_fit_summary_LOO_reject_SPclosest_MARCS.csv"
SED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GCOG and SPACE executable. Update .env after downloading/compiling SPACE locally.
GCOG_DIR = Path(os.getenv("GCOG_DIR", SPACE_DIR / "GCOGlibrarySPACE1")).resolve()
SPACE_EXECUTABLE_CANDIDATES = [
    Path(os.getenv("SPACE_EXECUTABLE", SPACE_DIR / "SPACE")).resolve(),
    SPACE_DIR / "SPACE",
    SPACE_DIR / "SPACE.exe",
    SPACE_DIR / "SPACE-1.4" / "SPACE",
    SPACE_DIR / "SPACE-1.4" / "SPACE.exe",
]
SPACE_EXECUTABLE = next((p for p in SPACE_EXECUTABLE_CANDIDATES if p.exists()), None)

# Control switches
RUN_STAGE0_FROM_FITS = True          # set False if aat_*.txt already exist
RUN_SPACE_STANDARD = True            # set False to only ingest existing model files
RUN_SPACE_ALPHA_METALS = True        # set False to only ingest existing model files
RUN_SED_FITTING = True               # set False if SED plots/summary already exist
MAKE_INTEGRATED_PLOTS = True

# For testing, set MAX_FIBERS = 3. For the full run, set None.
MAX_FIBERS = None
SKIP_EXISTING_SED_PNG = False

# Constants
C_KMS = 299792.458
CAT_REST_A = np.array([8498.02, 8542.09, 8662.14], dtype=float)
CAT_NAMES = ["CaT1", "CaT2", "CaT3"]
SPACE_WINDOWS = [(8450.0, 8493.0), (8503.0, 8535.0), (8550.0, 8660.0), (8670.0, 8745.0)]

print("BASE_DIR:", BASE_DIR)
print("RAW_FITS_DIR:", RAW_FITS_DIR, RAW_FITS_DIR.exists())
print("OUT_DIR:", OUT_DIR)
print("PHOT_FILE:", PHOT_FILE, PHOT_FILE.exists())
print("SP_ACE_AVG_FILE:", SP_ACE_AVG_FILE, SP_ACE_AVG_FILE.exists())
print("MARCS_CACHE_FILE:", MARCS_CACHE_FILE, MARCS_CACHE_FILE.exists())
print("SPACE_EXECUTABLE:", SPACE_EXECUTABLE)

In [ ]:
# ============================================================
# GENERAL HELPERS
# ============================================================

from astropy.io import fits
from scipy.optimize import curve_fit
from scipy.special import voigt_profile


def parse_fiber_id_from_name(path):
    path = Path(path)
    patterns = [
        r"aat_(\d+)",
        r"fiber[_-]?(\d+)",
        r"Fiber[_-]?(\d+)",
        r"space_model(?:_alpha_metals)?_(\d+)_",
        r"space_TGM(?:_ABD|_alpha_metals)?_(\d+)_",
        r"SED_Fiber_(\d+)_",
        r"n6569_2_(\d+)",
        r"n6569_(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, path.name)
        if m:
            return int(m.group(1))
    nums = re.findall(r"\d+", path.stem)
    return int(nums[-1]) if nums else None


def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        return pd.read_csv(path, delim_whitespace=True, comment="#")


def find_fits_files(raw_dir):
    raw_dir = Path(raw_dir)
    patterns = ["*.fits", "*.fit", "*.fts", "*.FITS", "*.FIT", "*.FTS", "*.fits.gz", "*.fit.gz"]
    files = []
    for pat in patterns:
        files.extend(raw_dir.glob(pat))
    # Exclude output/repaired subdirectories by requiring direct children of raw_dir.
    files = [p for p in files if p.parent.resolve() == raw_dir.resolve()]
    return sorted(set(files), key=lambda p: str(p).lower())


def read_two_column_spectrum(path):
    arr = np.loadtxt(path)
    if arr.ndim != 2 or arr.shape[1] < 2:
        raise ValueError(f"Expected two-column spectrum: {path}")
    wave = arr[:,0].astype(float)
    flux = arr[:,1].astype(float)
    med = np.nanmedian(wave[np.isfinite(wave)])
    if med < 1e-3:
        wave = wave * 1e10       # m -> Angstrom
    elif med < 10:
        wave = wave * 1e4        # micron -> Angstrom
    elif 840 < med < 900:
        wave = wave * 10         # nm -> Angstrom
    return wave, flux


def write_aat_txt(path, wave_A, flux_norm):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    order = np.argsort(wave_A)
    np.savetxt(path, np.column_stack([wave_A[order], flux_norm[order]]), fmt=["%.4f", "%.6f"])

In [ ]:
# ============================================================
# FITS HEADER REPAIR + ROBUST FITS READER
# ============================================================
# Some AAT FITS files have illegal unquoted RA/DEC cards. This cell makes
# byte-for-byte same-length repaired copies by replacing bad RA/DEC cards with
# COMMENT cards. The originals are not modified.


def _replace_bad_card_80(card_bytes):
    """Replace bad RA/DEC cards with harmless COMMENT cards, preserving 80 bytes."""
    key = card_bytes[:8]
    if key in [b"RA      ", b"DEC     "] and b"=" in card_bytes[:12]:
        txt = b"COMMENT bad RA/DEC card removed by NGC6569 pipeline"
        return txt.ljust(80, b" ")[:80]
    return card_bytes


def make_repaired_fits_copy(infile, outdir=REPAIRED_FITS_DIR):
    infile = Path(infile)
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    outfile = outdir / infile.name

    raw = infile.read_bytes()
    # FITS cards are 80 bytes; process all aligned 80-byte chunks.
    chunks = []
    n_full = len(raw) // 80
    for i in range(n_full):
        card = raw[i*80:(i+1)*80]
        chunks.append(_replace_bad_card_80(card))
    tail = raw[n_full*80:]
    outfile.write_bytes(b"".join(chunks) + tail)
    return outfile


def repair_all_fits(raw_dir=RAW_FITS_DIR, outdir=REPAIRED_FITS_DIR, max_files=None):
    files = find_fits_files(raw_dir)
    if max_files is not None:
        files = files[:max_files]
    columns = ["fiberid", "original_fits", "repaired_fits", "repair_status", "repair_error"]
    rows = []
    if not files:
        print(f"No FITS files found in RAW_FITS_DIR={Path(raw_dir).resolve()}")
    for i, p in enumerate(files, start=1):
        if i == 1 or i % 25 == 0 or i == len(files):
            print(f"Repairing {i}/{len(files)}: {p.name}")
        try:
            out = make_repaired_fits_copy(p, outdir)
            rows.append({"fiberid": parse_fiber_id_from_name(p), "original_fits": str(p), "repaired_fits": str(out), "repair_status": "ok"})
        except Exception as e:
            rows.append({"fiberid": parse_fiber_id_from_name(p), "original_fits": str(p), "repaired_fits": "", "repair_status": "FAILED", "repair_error": repr(e)})
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(TABLE_DIR / "fits_header_repair_manifest.csv", index=False)
    return df


def make_wavelength_from_header(header, n_pix):
    crval = None
    cdelt = None
    crpix = 1.0
    for key in ["CRVAL1", "CRVAL"]:
        if key in header:
            crval = float(header[key]); break
    for key in ["CDELT1", "CD1_1", "CDELT"]:
        if key in header:
            cdelt = float(header[key]); break
    for key in ["CRPIX1", "CRPIX"]:
        if key in header:
            crpix = float(header[key]); break
    if crval is None or cdelt is None:
        raise ValueError("Missing CRVAL1/CDELT1 wavelength keywords")
    pix = np.arange(n_pix, dtype=float) + 1.0
    wave = crval + (pix - crpix) * cdelt
    med = np.nanmedian(wave)
    if 840 < med < 900:
        wave = wave * 10.0
    elif 0.84 < med < 0.90:
        wave = wave * 10000.0
    elif 8.4e-7 < med < 8.9e-7:
        wave = wave * 1.0e10
    return wave


def read_fits_spectrum(fits_path):
    """Read one repaired AAT FITS file and return wavelength Angstrom, flux, header."""
    fits_path = Path(fits_path)
    with fits.open(fits_path, memmap=False, ignore_missing_end=True, lazy_load_hdus=False) as hdul:
        try:
            hdul.verify("silentfix")
        except Exception:
            pass
        # Prefer table HDU with wavelength/flux columns.
        for hdu in hdul:
            data = hdu.data
            if data is None:
                continue
            if hasattr(data, "names") and data.names is not None:
                names = list(data.names)
                lower = {n.lower(): n for n in names}
                wave_col = next((lower[c] for c in ["wave", "wavelength", "lambda", "lam", "wl"] if c in lower), None)
                flux_col = next((lower[c] for c in ["flux", "spec", "spectrum", "intensity", "counts"] if c in lower), None)
                if wave_col and flux_col:
                    wave = np.asarray(data[wave_col], dtype=float)
                    flux = np.asarray(data[flux_col], dtype=float)
                    good = np.isfinite(wave) & np.isfinite(flux)
                    return wave[good], flux[good], hdu.header
        # Otherwise use first image HDU with data and WCS.
        for hdu in hdul:
            data = hdu.data
            if data is None:
                continue
            arr = np.asarray(data, dtype=float)
            if arr.ndim == 1:
                flux = arr
            elif arr.ndim == 2:
                if arr.shape[0] == 2 and np.nanmedian(arr[0]) > 100:
                    wave, flux = arr[0], arr[1]
                    good = np.isfinite(wave) & np.isfinite(flux)
                    return wave[good], flux[good], hdu.header
                # choose row with most finite values
                finite_counts = np.sum(np.isfinite(arr), axis=1)
                flux = arr[np.argmax(finite_counts)]
            else:
                continue
            wave = make_wavelength_from_header(hdu.header, len(flux))
            good = np.isfinite(wave) & np.isfinite(flux)
            return wave[good], flux[good], hdu.header
    raise ValueError(f"Could not find readable spectrum in {fits_path.name}")

In [ ]:
# ============================================================
# NORMALIZATION + CaT VOIGT FITS + BARYCENTRIC RV PLACEHOLDER
# ============================================================

from numpy.polynomial.chebyshev import Chebyshev

CONTINUUM_WINDOWS = [(8400, 8480), (8510, 8530), (8570, 8640), (8690, 8780), (8820, 8840), (8845, 8875)]


def continuum_mask(wave_A, windows=CONTINUUM_WINDOWS):
    m = np.zeros_like(wave_A, dtype=bool)
    for lo, hi in windows:
        m |= (wave_A > lo) & (wave_A < hi)
    return m & np.isfinite(wave_A)


def normalize_spectrum_chebyshev(wave_A, flux, degree=4):
    wave_A = np.asarray(wave_A, dtype=float)
    flux = np.asarray(flux, dtype=float)
    good = continuum_mask(wave_A) & np.isfinite(flux) & (flux > 0)
    if good.sum() < degree + 5:
        raise ValueError("Too few continuum points")
    domain = [np.nanmin(wave_A[good]), np.nanmax(wave_A[good])]
    model0 = Chebyshev.fit(wave_A[good], flux[good], deg=degree, domain=domain)
    cont0 = model0(wave_A)
    norm0 = flux / cont0
    good2 = good & np.isfinite(norm0) & (norm0 > 0.75) & (norm0 < 1.25)
    if good2.sum() < degree + 5:
        good2 = good
    model1 = Chebyshev.fit(wave_A[good2], flux[good2], deg=degree, domain=domain)
    cont = model1(wave_A)
    norm = flux / cont
    return norm, cont, good2


def voigt_absorption_model(x, center, amp, sigma, gamma, slope, intercept):
    prof = voigt_profile(x - center, sigma, gamma)
    prof = prof / np.nanmax(prof) if np.nanmax(prof) > 0 else prof
    return intercept + slope * (x - center) - amp * prof


def fit_one_cat_line_observed(wave_A, flux_norm, rest_A, half_width_A=7.0):
    # Initial search allows large RV shift, then fit local line.
    m = np.isfinite(wave_A) & np.isfinite(flux_norm) & (wave_A > rest_A - half_width_A) & (wave_A < rest_A + half_width_A)
    if m.sum() < 15:
        return {"center_A": np.nan, "EW_A": np.nan, "RV_kms": np.nan, "fit_ok": False}
    x = wave_A[m]
    y = flux_norm[m]
    center0 = x[np.nanargmin(y)]
    depth = max(0.05, float(1.0 - np.nanmin(y)))
    p0 = [center0, depth, 0.25, 0.35, 0.0, 1.0]
    bounds = ([center0-1.5, 0.0, 0.03, 0.03, -0.2, 0.5], [center0+1.5, 1.5, 3.0, 4.0, 0.2, 1.5])
    try:
        popt, pcov = curve_fit(voigt_absorption_model, x, y, p0=p0, bounds=bounds, maxfev=20000)
        model = voigt_absorption_model(x, *popt)
        center, amp, sigma, gamma, slope, intercept = popt
        cont = intercept + slope * (x - center)
        ew = float(np.trapz(np.clip(1 - model / cont, -0.1, 2.0), x))
        rv = float((center - rest_A) / rest_A * C_KMS)
        rms = float(np.sqrt(np.nanmean((y - model)**2)))
        return {"center_A": center, "EW_A": ew, "RV_kms": rv, "fit_rms": rms, "fit_ok": True, "popt": popt}
    except Exception:
        return {"center_A": np.nan, "EW_A": np.nan, "RV_kms": np.nan, "fit_ok": False}


def fit_cat_triplet_observed(wave_A, flux_norm):
    linefits = [fit_one_cat_line_observed(wave_A, flux_norm, lam) for lam in CAT_REST_A]
    rvs = [d["RV_kms"] for d in linefits if d.get("fit_ok") and np.isfinite(d.get("RV_kms", np.nan))]
    rec = {}
    for i, d in enumerate(linefits, start=1):
        rec[f"CaT{i}_center_A"] = d.get("center_A", np.nan)
        rec[f"CaT{i}_EW"] = d.get("EW_A", np.nan)
        rec[f"CaT{i}_RV"] = d.get("RV_kms", np.nan)
        rec[f"CaT{i}_fit_ok"] = bool(d.get("fit_ok", False))
    rec["RVraw_CaT"] = float(np.nanmean(rvs)) if rvs else np.nan
    rec["e_RVraw_CaT"] = float(np.nanstd(rvs, ddof=1)) if len(rvs) > 1 else np.nan
    rec["CaT1"] = rec["CaT1_EW"]
    rec["CaT2"] = rec["CaT2_EW"]
    rec["CaT3"] = rec["CaT3_EW"]
    rec["SumEW23"] = rec["CaT2_EW"] + rec["CaT3_EW"] if np.isfinite(rec["CaT2_EW"]) and np.isfinite(rec["CaT3_EW"]) else np.nan
    rec["SumEW123"] = np.nansum([rec["CaT1_EW"], rec["CaT2_EW"], rec["CaT3_EW"]])
    return rec, linefits


def shift_to_lab_frame(wave_A, rvraw_kms):
    return wave_A / (1.0 + float(rvraw_kms) / C_KMS)


def estimate_snr_from_continuum(flux_norm, cont_mask):
    vals = np.asarray(flux_norm)[cont_mask]
    vals = vals[np.isfinite(vals)]
    if len(vals) < 10:
        return np.nan
    sigma = 1.4826 * np.nanmedian(np.abs(vals - np.nanmedian(vals)))
    return float(1.0/sigma) if sigma > 0 else np.nan


def apply_barycentric_correction(rvraw, header=None, phot_row=None):
    # For NGC 6569 legacy notebook use the fixed correction if header RA/DEC are unreliable.
    barycorr = -13.353688041694257
    return float(rvraw + barycorr) if np.isfinite(rvraw) else np.nan, barycorr, "fixed_NGC6569_legacy"

In [ ]:
# ============================================================
# STAGE 0: FITS -> NORMALIZED REST-FRAME aat_*.txt + CaT TABLE
# ============================================================

def process_one_fits_to_aat(fits_path, out_dir, phot_indexed=None):
    fits_path = Path(fits_path)
    fiberid = parse_fiber_id_from_name(fits_path)
    rec = {"fiberid": fiberid, "fits_file": str(fits_path), "status": "not_started", "error": ""}
    try:
        wave, flux, header = read_fits_spectrum(fits_path)
        flux_norm, cont, cmask = normalize_spectrum_chebyshev(wave, flux)
        cat, linefits = fit_cat_triplet_observed(wave, flux_norm)
        rvraw = cat.get("RVraw_CaT", np.nan)
        wave_lab = shift_to_lab_frame(wave, rvraw) if np.isfinite(rvraw) else wave.copy()
        snr = estimate_snr_from_continuum(flux_norm, cmask)
        phot_row = phot_indexed.loc[fiberid] if phot_indexed is not None and fiberid in phot_indexed.index else None
        rv_bary, barycorr, bary_mode = apply_barycentric_correction(rvraw, header=header, phot_row=phot_row)
        aat_path = Path(out_dir) / f"aat_{fiberid}.txt"
        write_aat_txt(aat_path, wave_lab, flux_norm)
        rec.update({
            "aat_txt_file": str(aat_path),
            "frame_in": "observed/topocentric",
            "frame_out": "stellar_rest_lab",
            "RVraw_CaT": rvraw,
            "barycorr_kms": barycorr,
            "barycorr_mode": bary_mode,
            "RV_bary_CaT": rv_bary,
            "SNR_norm_est": snr,
            "status": "ok",
            **cat,
        })
    except Exception as e:
        rec["status"] = "FAILED"
        rec["error"] = repr(e)
        print(f"FAILED {fits_path.name}: {e}")
    return rec


def run_stage0_from_fits():
    phot = read_table_auto(PHOT_FILE) if PHOT_FILE.exists() else pd.DataFrame()
    phot_indexed = phot.set_index("fiberid") if "fiberid" in phot.columns else None

    repair_manifest = repair_all_fits(RAW_FITS_DIR, REPAIRED_FITS_DIR, max_files=MAX_FIBERS)
    repaired_files = [Path(p) for p in repair_manifest.loc[repair_manifest["repair_status"] == "ok", "repaired_fits"].dropna()]
    if not repaired_files:
        raise FileNotFoundError(f"No repaired FITS files are available. Check RAW_FITS_DIR={RAW_FITS_DIR}")
    if MAX_FIBERS is not None:
        repaired_files = repaired_files[:MAX_FIBERS]

    rows = []
    for i, p in enumerate(repaired_files, start=1):
        if i == 1 or i % 25 == 0 or i == len(repaired_files):
            print(f"Stage 0 {i}/{len(repaired_files)}: {p.name}")
        rows.append(process_one_fits_to_aat(p, NORMALIZED_AAT_DIR, phot_indexed=phot_indexed))

    cat_master = pd.DataFrame(rows)
    if "fiberid" in cat_master.columns and cat_master["fiberid"].notna().any():
        cat_master = cat_master.sort_values("fiberid").reset_index(drop=True)
    out_csv = TABLE_DIR / "CaT_master_from_FITS_normalized_labframe.csv"
    cat_master.to_csv(out_csv, index=False)
    print("Wrote", out_csv)
    print(cat_master["status"].value_counts(dropna=False))
    return cat_master

if RUN_STAGE0_FROM_FITS:
    cat_master = run_stage0_from_fits()
else:
    cat_path = TABLE_DIR / "CaT_master_from_FITS_normalized_labframe.csv"
    if cat_path.exists():
        cat_master = pd.read_csv(cat_path)
    elif PHOT_FILE.exists():
        # Fallback to an existing CaT/photometry table if you are not rerunning FITS preprocessing.
        cat_master = pd.read_csv(PHOT_FILE)
    else:
        cat_master = pd.DataFrame()

cat_master.head()

In [ ]:
# ============================================================
# STAGE 1: DIAS & PARISI (2020) CaT [Fe/H]DP
# ============================================================
# Uses raw apparent BDBS imag, not dereddened i0.
# Prompts for the local HB/clump level in apparent i.

try:
    imag_HB
except NameError:
    imag_HB = float(input("Enter local apparent imag_HB for this field, e.g. 16.59: "))

BETA_I = 0.62
BETA_I_ERR = np.sqrt(0.06**2 + 0.17**2)
DP_INTERCEPT = -2.966
DP_SLOPE = 0.362

phot = pd.read_csv(PHOT_FILE) if PHOT_FILE.exists() else pd.DataFrame()
work = cat_master.copy()
if "fiberid" in work.columns and "fiberid" in phot.columns:
    keep_cols = [c for c in ["fiberid", "imag", "ierr"] if c in phot.columns]
    work = work.merge(phot[keep_cols], on="fiberid", how="left", suffixes=("", "_phot"))

# Prefer measured 3-line sum. If unavailable, convert EW2L to 3-line scale using Dias/Parisi-style relation.
if "SumEW123" not in work.columns:
    work["SumEW123"] = np.nan
if "SumEW23" not in work.columns:
    if "CaT2" in work.columns and "CaT3" in work.columns:
        work["SumEW23"] = work["CaT2"] + work["CaT3"]
    else:
        work["SumEW23"] = np.nan

work["EW3L_for_DP"] = work["SumEW123"]
missing3 = ~np.isfinite(pd.to_numeric(work["EW3L_for_DP"], errors="coerce"))
work.loc[missing3, "EW3L_for_DP"] = 1.26 + 1.00 * pd.to_numeric(work.loc[missing3, "SumEW23"], errors="coerce")

work["Wi_prime_DP"] = work["EW3L_for_DP"] + BETA_I * (pd.to_numeric(work["imag"], errors="coerce") - imag_HB)
work["FeH_CaT_DP"] = DP_INTERCEPT + DP_SLOPE * work["Wi_prime_DP"]
work["e_FeH_CaT_DP_beta_only"] = abs(DP_SLOPE * (pd.to_numeric(work["imag"], errors="coerce") - imag_HB) * BETA_I_ERR)
work["imag_HB_used"] = imag_HB

cat_dp_table = work
cat_dp_path = TABLE_DIR / "CaT_master_with_Dias_Parisi_FeHDP.csv"
cat_dp_table.to_csv(cat_dp_path, index=False)
print("Wrote", cat_dp_path)
cat_dp_table[["fiberid", "imag", "EW3L_for_DP", "Wi_prime_DP", "FeH_CaT_DP", "RV_bary_CaT"]].head()

In [ ]:
# ============================================================
# STAGE 2: RUN OR INGEST SPACE/SP_Ace STANDARD + ALPHA/METALS MODELS
# ============================================================


def write_space_par(par_path, obs_sp_file, alpha=False, fwhm=0.8):
    lines = [
        f"obs_sp_file '{obs_sp_file}'",
        f"GCOGlib '{str(GCOG_DIR) + os.sep}'",
        f"fwhm {fwhm}",
        "wave_lims 8450 8493 8503 8535 8550 8660 8670 8745",
        "no_norm",
        "error_est",
        "null_value 'NaN'",
        "alpha" if alpha else "#alpha",
        "ABD_loop",
        "Salaris_MH",
    ]
    par_path = Path(par_path)
    par_path.parent.mkdir(parents=True, exist_ok=True)
    par_path.write_text("\n".join(lines) + "\n")


def run_space_for_one_aat(aat_path, mode="standard"):
    aat_path = Path(aat_path)
    fid = parse_fiber_id_from_name(aat_path)
    alpha = (mode == "alpha_metals")
    run_dir = SPACE_RUN_DIR / mode / f"fiber_{fid}"
    run_dir.mkdir(parents=True, exist_ok=True)
    local_aat = run_dir / aat_path.name
    shutil.copy2(aat_path, local_aat)
    par_path = run_dir / "space.par"
    write_space_par(par_path, local_aat, alpha=alpha)

    if SPACE_EXECUTABLE is None or not Path(SPACE_EXECUTABLE).exists():
        return {"fiberid": fid, "mode": mode, "status": "SKIPPED_no_SPACE_executable", "run_dir": str(run_dir)}

    try:
        proc = subprocess.run([str(SPACE_EXECUTABLE)], cwd=str(run_dir), text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=600)
        status = "ok" if proc.returncode == 0 else f"FAILED_returncode_{proc.returncode}"
        # Copy standard output filenames into canonical output names.
        model_in = run_dir / "space_model.dat"
        tgm_abd = run_dir / "space_TGM_ABD.dat"
        tgm = run_dir / "space_TGM.dat"
        model_out = SPACE_OUTPUT_DIR / (f"space_model_alpha_metals_{fid}_best_conservative_no_norm.dat" if alpha else f"space_model_{fid}_best_conservative_no_norm.dat")
        tgm_out = SPACE_OUTPUT_DIR / (f"space_TGM_alpha_metals_{fid}_best_conservative_no_norm.dat" if alpha else f"space_TGM_ABD_{fid}_best_conservative_no_norm.dat")
        if model_in.exists(): shutil.copy2(model_in, model_out)
        if tgm_abd.exists(): shutil.copy2(tgm_abd, tgm_out)
        elif tgm.exists(): shutil.copy2(tgm, tgm_out)
        return {"fiberid": fid, "mode": mode, "status": status, "run_dir": str(run_dir), "stdout": proc.stdout[-2000:], "stderr": proc.stderr[-2000:]}
    except Exception as e:
        return {"fiberid": fid, "mode": mode, "status": "FAILED", "run_dir": str(run_dir), "error": repr(e)}


def run_space_batch():
    aat_files = sorted(NORMALIZED_AAT_DIR.glob("aat_*.txt"), key=lambda p: parse_fiber_id_from_name(p) or -1)
    if MAX_FIBERS is not None:
        aat_files = aat_files[:MAX_FIBERS]
    rows = []
    for mode, do_run in [("standard", RUN_SPACE_STANDARD), ("alpha_metals", RUN_SPACE_ALPHA_METALS)]:
        if not do_run:
            continue
        print(f"Running mode: {mode}")
        for i, p in enumerate(aat_files, start=1):
            if i == 1 or i % 25 == 0 or i == len(aat_files):
                print(f"  {i}/{len(aat_files)} {p.name}")
            rows.append(run_space_for_one_aat(p, mode=mode))
    log = pd.DataFrame(rows)
    log.to_csv(TABLE_DIR / "SPACE_run_log.csv", index=False)
    return log

space_run_log = run_space_batch() if (RUN_SPACE_STANDARD or RUN_SPACE_ALPHA_METALS) else pd.DataFrame()
space_run_log.head()

## Stage 3: MARCS SED fitting using the original `SEDfit_v2` logic

The following cell is intentionally based on the original `SEDfit_v2` leave-one-out MARCS-cache fitting cell. It makes the same kind of SED plots as your existing code: blue best photometric MARCS model, orange SP_Ace-nearest MARCS model, BDBS dereddened photometry, and rejected-band marking. It writes one PNG per fiber plus the SED summary table.

In [ ]:
# ------------------------------------------------------------
# FULL RESTRICTED BATCH MARCS SED FITTING CELL
# with leave-one-out photometry outlier rejection
# and MARCS model closest to the SP_Ace fit overplotted
#
# Plot layout:
#   legend fixed inside the plot at upper-left
#   information box fixed inside the plot at lower-left
#
# Outputs:
#   1. Master table:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_batch_restricted_fit_summary_LOO_reject_SPclosest_MARCS.csv
#   2. One PNG per modeled fiber in:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_SED_PLOTS_RESTRICTED_LOO_REJECT_SPclosest_MARCS
# ------------------------------------------------------------

import os
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

phot_file = str(PHOT_FILE)

summary_file_candidates = [
    str(SP_ACE_AVG_FILE),
    str(BASE_DIR / "SP_Ace_averages_with_FeHDP_avg_std(1).csv"),
    str(BASE_DIR / "SP_Ace_averages_with_FeHDP_avg_std(2).csv"),
]

MARCS_DIR = str(MARCS_DIR)
cache_file = str(MARCS_CACHE_FILE)

output_dir = str(SED_OUTPUT_DIR)
os.makedirs(output_dir, exist_ok=True)

master_output_csv = str(SED_SUMMARY_CSV)

# For testing, set this to e.g. 10.
# For the full run, set to None.
max_fibers_to_run = MAX_FIBERS

make_plots = True
skip_existing_png = SKIP_EXISTING_SED_PNG

# Manual exclusions, if needed.
exclude_bands_global = []

per_fiber_exclude_bands = {
    # Example:
    # 69: ["r"],
    # 215: ["r"],
}

# Formal BDBS errors are often too small for model/photometry comparison.
phot_error_floor_mag = 0.05

# ------------------------------------------------------------
# Improved automatic photometry outlier rejection
# ------------------------------------------------------------

enable_photometry_outlier_rejection = True

# Candidate point must be this discrepant from the leave-one-out refit.
phot_outlier_sigma = 3.0

# Reject at most this many bands automatically.
phot_outlier_max_reject = 1

# Do not reject if too few bands would remain.
min_bands_after_rejection = 4

# Require the leave-one-out reduced chi2 to improve enough.
phot_outlier_min_redchi2_improvement = 1.0
phot_outlier_min_redchi2_ratio = 0.80

# Require a local SED kink if possible.
# This helps avoid rejecting a point that is part of a broad color mismatch.
phot_outlier_local_mag_threshold = 0.30

# Flag possible variables / broad photometric mismatch after final fit.
variable_flag_abs_resid_sigma = 3.0
variable_flag_min_bad_bands = 2
variable_flag_red_chi2 = 5.0

# MARCS display smoothing.
# 120 = smooth SED shape.
# 1800 = keeps more spectral structure visible.
model_smoothing_bins = 1800

# SP_Ace prior widths for the restricted photometric fit.
teff_half_width_default = 600
logg_half_width_default = 1.0
feh_half_width_default = 0.50
alpha_half_width_default = 0.40

# Low-S/N widening.
low_snr_threshold = 25
teff_half_width_lowsnr = 800
logg_half_width_lowsnr = 1.25
feh_half_width_lowsnr = 0.60
alpha_half_width_lowsnr = 0.50

# Optional cache filters before CMD/SP_Ace restrictions.
abundance_filter = None     # None, "st", or "ae"
geometry_filter = None      # None, "s", or "p"

# If a fiber has zero models after cuts, set this True to retry with wider cuts.
allow_fallback_wider_box = False

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_eff_um, band_min_um, band_max_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 0.32, 0.40, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 0.40, 0.55, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 0.55, 0.70, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 0.70, 0.85, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 0.85, 0.96, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 0.96, 1.08, 3631.0),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def find_existing_file(candidates):
    for f in candidates:
        if os.path.exists(f):
            return f
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not numeric data:\n{path}")

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


def resolve_model_path(path_from_cache, filename, marcs_dir):
    if isinstance(path_from_cache, str) and os.path.exists(path_from_cache):
        return path_from_cache

    matches = glob.glob(os.path.join(marcs_dir, "**", filename), recursive=True)

    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find MARCS model file:\n{filename}\nCached path was:\n{path_from_cache}"
    )


# ------------------------------------------------------------
# MARCS MODEL HELPERS
# ------------------------------------------------------------

def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def bin_model_in_log_lambda(lambda_um, flux, n_bins=1800):
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    if len(lambda_um) < 5:
        return lambda_um, flux

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lam_b = []
    flux_b = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lam_b.append(10 ** np.nanmean(loglam[m]))
        flux_b.append(np.nanmedian(flux[m]))

    return np.array(lam_b), np.array(flux_b)


# ------------------------------------------------------------
# OBSERVED SED HELPERS
# ------------------------------------------------------------

def build_dereddened_sed(row, exclude_bands=None):
    if exclude_bands is None:
        exclude_bands = []

    sed_rows = []

    for band, mag_col, err_col, A_col, lam_eff, band_min, band_max, zp_jy in bands:

        if band in exclude_bands:
            continue

        if mag_col not in row.index:
            continue

        mag = pd.to_numeric(row[mag_col], errors="coerce")
        emag_formal = pd.to_numeric(row[err_col], errors="coerce") if err_col in row.index else np.nan
        A_lambda = pd.to_numeric(row[A_col], errors="coerce") if A_col in row.index else 0.0

        if not np.isfinite(mag):
            continue
        if mag > 90 or mag < -90:
            continue
        if not np.isfinite(A_lambda):
            A_lambda = 0.0

        mag0 = mag - A_lambda

        if np.isfinite(emag_formal) and emag_formal > 0:
            emag_fit = np.sqrt(emag_formal**2 + phot_error_floor_mag**2)
        else:
            emag_fit = phot_error_floor_mag

        fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
        fnu_cgs = fnu_jy * 1.0e-23

        lam_cm = lam_eff * 1.0e-4
        nu_hz = c_cgs / lam_cm
        nu_fnu = nu_hz * fnu_cgs

        frac_err = 0.4 * np.log(10.0) * emag_fit
        nu_fnu_err = nu_fnu * frac_err

        sed_rows.append({
            "band": band,
            "lambda_um": lam_eff,
            "mag_obs": mag,
            "A_lambda": A_lambda,
            "mag0": mag0,
            "mag_err_fit": emag_fit,
            "nu_fnu": nu_fnu,
            "nu_fnu_err": nu_fnu_err,
            "model_col": f"model_{band}",
        })

    sed = pd.DataFrame(sed_rows)

    if len(sed) > 0:
        sed = sed.sort_values("lambda_um").reset_index(drop=True)

    return sed


def get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp):
    u0 = pd.to_numeric(row["umag"], errors="coerce") - pd.to_numeric(row["Au"], errors="coerce")
    i0 = pd.to_numeric(row["imag"], errors="coerce") - pd.to_numeric(row["Ai"], errors="coerce")
    ui0 = u0 - i0

    if np.isfinite(ui0) and np.isfinite(i0) and (ui0 >= 3.8) and (i0 <= 14.5):
        cmd_region = "bright RGB/AGB"
        teff_min, teff_max = 3300, 4800
        logg_min, logg_max = -0.5, 2.5

    elif np.isfinite(ui0) and (ui0 >= 3.0):
        cmd_region = "RGB"
        teff_min, teff_max = 3500, 5200
        logg_min, logg_max = 0.0, 3.2

    elif np.isfinite(ui0) and np.isfinite(i0) and (2.0 <= ui0 < 3.8) and (14.0 <= i0 <= 16.8):
        cmd_region = "red clump / red HB / lower RGB"
        teff_min, teff_max = 4000, 6000
        logg_min, logg_max = 1.0, 3.8

    elif np.isfinite(ui0) and np.isfinite(i0) and (ui0 < 2.0) and (14.5 <= i0 <= 17.0):
        cmd_region = "blue HB / warmer star"
        teff_min, teff_max = 5500, 8500
        logg_min, logg_max = 2.0, 4.8

    else:
        cmd_region = "broad fallback"
        teff_min, teff_max = 3300, 7000
        logg_min, logg_max = -0.5, 4.5

    teff_half_width = teff_half_width_default
    logg_half_width = logg_half_width_default
    feh_half_width = feh_half_width_default
    alpha_half_width = alpha_half_width_default

    snr_here = pd.to_numeric(row["S/N"], errors="coerce") if "S/N" in row.index else np.nan

    if np.isfinite(snr_here) and snr_here < low_snr_threshold:
        teff_half_width = teff_half_width_lowsnr
        logg_half_width = logg_half_width_lowsnr
        feh_half_width = feh_half_width_lowsnr
        alpha_half_width = alpha_half_width_lowsnr

    teff_low = max(teff_min, teff_sp - teff_half_width)
    teff_high = min(teff_max, teff_sp + teff_half_width)

    logg_low = max(logg_min, logg_sp - logg_half_width)
    logg_high = min(logg_max, logg_sp + logg_half_width)

    feh_low = feh_sp - feh_half_width
    feh_high = feh_sp + feh_half_width

    alpha_low = afe_sp - alpha_half_width
    alpha_high = afe_sp + alpha_half_width

    return {
        "u0": u0,
        "i0": i0,
        "ui0": ui0,
        "cmd_region": cmd_region,

        "teff_low": teff_low,
        "teff_high": teff_high,
        "logg_low": logg_low,
        "logg_high": logg_high,
        "feh_low": feh_low,
        "feh_high": feh_high,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,

        "teff_min_cmd": teff_min,
        "teff_max_cmd": teff_max,
        "logg_min_cmd": logg_min,
        "logg_max_cmd": logg_max,

        "teff_half_width": teff_half_width,
        "logg_half_width": logg_half_width,
        "feh_half_width": feh_half_width,
        "alpha_half_width": alpha_half_width,
    }


def restrict_grid_for_fiber(grid_base, teff_sp, logg_sp, feh_sp, afe_sp, cmd_info):
    grid = grid_base.copy()

    grid = grid[
        (grid["teff"] >= cmd_info["teff_low"])
        & (grid["teff"] <= cmd_info["teff_high"])
        & (grid["logg"] >= cmd_info["logg_low"])
        & (grid["logg"] <= cmd_info["logg_high"])
        & (grid["feh"] >= cmd_info["feh_low"])
        & (grid["feh"] <= cmd_info["feh_high"])
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= cmd_info["alpha_low"])
            & (grid["alpha"] <= cmd_info["alpha_high"])
        ].copy()

    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    return grid.reset_index(drop=True), used_geom


def restrict_grid_fallback(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    grid = grid_base[
        (grid_base["teff"] >= teff_sp - 1000)
        & (grid_base["teff"] <= teff_sp + 1000)
        & (grid_base["logg"] >= logg_sp - 1.5)
        & (grid_base["logg"] <= logg_sp + 1.5)
        & (grid_base["feh"] >= feh_sp - 0.75)
        & (grid_base["feh"] <= feh_sp + 0.75)
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= afe_sp - 0.6)
            & (grid["alpha"] <= afe_sp + 0.6)
        ].copy()

    return grid.reset_index(drop=True)


# ------------------------------------------------------------
# FITTING HELPERS
# ------------------------------------------------------------

def vectorized_cached_fit(grid, sed):
    obs_flux = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)
    model_cols = sed["model_col"].tolist()

    missing_cols = [c for c in model_cols if c not in grid.columns]
    if len(missing_cols) > 0:
        raise KeyError(f"Cache is missing model columns: {missing_cols}")

    M = grid[model_cols].to_numpy(dtype=float)

    y = obs_flux.astype(float)
    s = obs_err.astype(float)

    good_band = np.isfinite(y) & np.isfinite(s) & (y > 0) & (s > 0)

    if np.sum(good_band) < 2:
        raise ValueError("Not enough valid observed bands for SED fit.")

    M = M[:, good_band]
    y = y[good_band]
    s = s[good_band]

    good_model = np.all(np.isfinite(M) & (M > 0), axis=1)

    grid_fit = grid.loc[good_model].copy().reset_index(drop=True)
    M = M[good_model, :]

    if len(grid_fit) == 0:
        raise ValueError("No valid model rows after checking cached model fluxes.")

    w = 1.0 / s**2

    numerator = np.sum(M * y * w, axis=1)
    denominator = np.sum(M**2 * w, axis=1)

    scale = numerator / denominator

    resid = y[None, :] - scale[:, None] * M
    chi2 = np.sum((resid / s[None, :])**2, axis=1)

    nfit = int(np.sum(good_band))
    dof = nfit - 1

    if dof > 0:
        red_chi2 = chi2 / dof
    else:
        red_chi2 = np.full_like(chi2, np.nan)

    results_df = grid_fit.copy()
    results_df["scale"] = scale
    results_df["chi2"] = chi2
    results_df["red_chi2"] = red_chi2
    results_df["nfit"] = nfit

    results_df = results_df[np.isfinite(results_df["chi2"])].copy()
    results_df = results_df.sort_values("chi2").reset_index(drop=True)

    if len(results_df) == 0:
        raise ValueError("No valid cached fits produced.")

    return results_df


def compute_best_band_residuals(best, sed):
    model_cols = sed["model_col"].tolist()

    obs = sed["nu_fnu"].to_numpy(dtype=float)
    err = sed["nu_fnu_err"].to_numpy(dtype=float)

    model_raw = best[model_cols].to_numpy(dtype=float)
    model_scaled = best["scale"] * model_raw

    residual = obs - model_scaled
    residual_sigma = residual / err
    frac_residual = residual / model_scaled

    mag_residual = np.full_like(residual, np.nan, dtype=float)
    good = (obs > 0) & (model_scaled > 0)

    # Positive mag_residual means observed point is fainter/lower than model.
    mag_residual[good] = -2.5 * np.log10(obs[good] / model_scaled[good])

    resid_df = sed[["band", "lambda_um", "mag_obs", "mag0", "mag_err_fit"]].copy()
    resid_df["obs_nu_fnu"] = obs
    resid_df["model_nu_fnu"] = model_scaled
    resid_df["residual_nu_fnu"] = residual
    resid_df["residual_sigma"] = residual_sigma
    resid_df["frac_residual"] = frac_residual
    resid_df["mag_residual_obs_minus_model"] = mag_residual
    resid_df["abs_residual_sigma"] = np.abs(residual_sigma)

    resid_df = resid_df.sort_values("abs_residual_sigma", ascending=False).reset_index(drop=True)

    return resid_df


def local_sed_isolation_mag(sed, band):
    """
    Measure whether a band is locally isolated relative to its neighbors.
    Positive value means the point is low/faint relative to local interpolation.
    Negative value means the point is high/bright relative to local interpolation.
    """

    sed_sorted = sed.sort_values("lambda_um").reset_index(drop=True)

    idx_list = sed_sorted.index[sed_sorted["band"] == band].tolist()

    if len(idx_list) == 0:
        return np.nan

    idx = idx_list[0]

    # Need one neighbor on each side.
    if idx == 0 or idx == len(sed_sorted) - 1:
        return np.nan

    left = sed_sorted.iloc[idx - 1]
    mid = sed_sorted.iloc[idx]
    right = sed_sorted.iloc[idx + 1]

    if (
        left["nu_fnu"] <= 0
        or mid["nu_fnu"] <= 0
        or right["nu_fnu"] <= 0
        or left["lambda_um"] <= 0
        or mid["lambda_um"] <= 0
        or right["lambda_um"] <= 0
    ):
        return np.nan

    x_left = np.log10(left["lambda_um"])
    x_mid = np.log10(mid["lambda_um"])
    x_right = np.log10(right["lambda_um"])

    y_left = np.log10(left["nu_fnu"])
    y_mid = np.log10(mid["nu_fnu"])
    y_right = np.log10(right["nu_fnu"])

    y_pred = y_left + (y_right - y_left) * (x_mid - x_left) / (x_right - x_left)

    delta_mag = -2.5 * (y_mid - y_pred)

    return delta_mag


def evaluate_leave_one_out_candidate(grid, sed_work, band_to_test, current_red_chi2):
    """
    Try removing one band and refit.

    This asks:
      1. Does removing this band improve reduced chi2?
      2. Is the removed band discrepant from the leave-one-out refit?
      3. Is the removed band locally suspicious relative to neighboring bands?
    """

    sed_try = sed_work[sed_work["band"] != band_to_test].copy().reset_index(drop=True)

    if len(sed_try) < min_bands_after_rejection:
        return None

    results_try = vectorized_cached_fit(grid, sed_try)
    best_try = results_try.iloc[0]

    cand = sed_work[sed_work["band"] == band_to_test].iloc[0]

    obs = float(cand["nu_fnu"])
    err = float(cand["nu_fnu_err"])
    model_col = cand["model_col"]

    if (
        not np.isfinite(obs)
        or not np.isfinite(err)
        or err <= 0
        or model_col not in best_try.index
        or not np.isfinite(best_try[model_col])
        or best_try[model_col] <= 0
    ):
        return None

    pred = float(best_try["scale"] * best_try[model_col])

    if not np.isfinite(pred) or pred <= 0:
        return None

    residual_sigma = (obs - pred) / err
    abs_residual_sigma = abs(residual_sigma)

    # Positive means observed point is fainter/lower than the leave-one-out model.
    mag_resid = -2.5 * np.log10(obs / pred)

    local_delta_mag = local_sed_isolation_mag(sed_work, band_to_test)

    red_try = float(best_try["red_chi2"])
    red_improvement = current_red_chi2 - red_try

    if np.isfinite(current_red_chi2) and current_red_chi2 > 0 and np.isfinite(red_try):
        red_ratio = red_try / current_red_chi2
    else:
        red_ratio = np.nan

    if np.isfinite(local_delta_mag):
        locally_suspicious = abs(local_delta_mag) >= phot_outlier_local_mag_threshold
    else:
        # End bands have no two-sided local check, so require stronger evidence.
        locally_suspicious = abs_residual_sigma >= 5.0

    chi2_improves_enough = (
        np.isfinite(red_improvement)
        and np.isfinite(red_ratio)
        and (
            red_improvement >= phot_outlier_min_redchi2_improvement
            or red_ratio <= phot_outlier_min_redchi2_ratio
        )
    )

    passes_rejection_test = (
        abs_residual_sigma >= phot_outlier_sigma
        and chi2_improves_enough
        and locally_suspicious
    )

    local_weight = abs(local_delta_mag) if np.isfinite(local_delta_mag) else 0.0

    score = abs_residual_sigma * max(red_improvement, 0.0) * (1.0 + local_weight)

    return {
        "band": band_to_test,
        "sed_try": sed_try,
        "results_try": results_try,
        "best_try": best_try,
        "red_chi2_try": red_try,
        "red_chi2_current": current_red_chi2,
        "red_chi2_improvement": red_improvement,
        "red_chi2_ratio": red_ratio,
        "residual_sigma_leave_one_out": residual_sigma,
        "abs_residual_sigma_leave_one_out": abs_residual_sigma,
        "mag_residual_leave_one_out": mag_resid,
        "local_delta_mag": local_delta_mag,
        "locally_suspicious": locally_suspicious,
        "chi2_improves_enough": chi2_improves_enough,
        "passes_rejection_test": passes_rejection_test,
        "score": score,
    }


def fit_with_photometry_outlier_rejection(grid, sed_initial):
    """
    Improved automatic photometry rejection.

    Instead of rejecting the largest residual from a single global fit,
    this tests each band by leave-one-out refitting.
    """

    sed_work = sed_initial.copy().reset_index(drop=True)

    rejected_bands = []
    rejection_reasons = []
    rejection_diagnostics = []

    for rejection_pass in range(phot_outlier_max_reject + 1):

        results_current = vectorized_cached_fit(grid, sed_work)
        best_current = results_current.iloc[0]
        current_red_chi2 = float(best_current["red_chi2"])

        if not enable_photometry_outlier_rejection:
            break

        if len(rejected_bands) >= phot_outlier_max_reject:
            break

        if (len(sed_work) - 1) < min_bands_after_rejection:
            break

        candidates = []

        for band in sed_work["band"].tolist():
            cand = evaluate_leave_one_out_candidate(
                grid=grid,
                sed_work=sed_work,
                band_to_test=band,
                current_red_chi2=current_red_chi2
            )

            if cand is not None:
                candidates.append(cand)

        if len(candidates) == 0:
            break

        cand_df = pd.DataFrame([
            {
                "band": c["band"],
                "red_chi2_current": c["red_chi2_current"],
                "red_chi2_try": c["red_chi2_try"],
                "red_chi2_improvement": c["red_chi2_improvement"],
                "red_chi2_ratio": c["red_chi2_ratio"],
                "residual_sigma_leave_one_out": c["residual_sigma_leave_one_out"],
                "abs_residual_sigma_leave_one_out": c["abs_residual_sigma_leave_one_out"],
                "mag_residual_leave_one_out": c["mag_residual_leave_one_out"],
                "local_delta_mag": c["local_delta_mag"],
                "locally_suspicious": c["locally_suspicious"],
                "chi2_improves_enough": c["chi2_improves_enough"],
                "passes_rejection_test": c["passes_rejection_test"],
                "score": c["score"],
            }
            for c in candidates
        ])

        rejection_diagnostics.append(cand_df)

        passing = [c for c in candidates if c["passes_rejection_test"]]

        if len(passing) == 0:
            break

        best_candidate = sorted(passing, key=lambda c: c["score"], reverse=True)[0]

        reject_band = best_candidate["band"]
        rejected_bands.append(reject_band)

        direction = "high" if best_candidate["residual_sigma_leave_one_out"] > 0 else "low"

        reason = (
            f"{reject_band}: {direction}, "
            f"LOO_resid_sigma={best_candidate['residual_sigma_leave_one_out']:.2f}, "
            f"local_delta_mag={best_candidate['local_delta_mag']:.2f}, "
            f"red_chi2 {best_candidate['red_chi2_current']:.2f} -> {best_candidate['red_chi2_try']:.2f}"
        )

        rejection_reasons.append(reason)

        sed_work = sed_work[sed_work["band"] != reject_band].copy().reset_index(drop=True)

    final_results_df = vectorized_cached_fit(grid, sed_work)
    final_best = final_results_df.iloc[0]
    final_resid_df = compute_best_band_residuals(final_best, sed_work)

    if len(rejection_diagnostics) > 0:
        last_candidate_table = rejection_diagnostics[-1].copy()
        last_candidate_table = last_candidate_table.sort_values("score", ascending=False).reset_index(drop=True)
    else:
        last_candidate_table = pd.DataFrame()

    return {
        "results_df": final_results_df,
        "best": final_best,
        "sed_used": sed_work,
        "sed_initial": sed_initial,
        "rejected_bands": rejected_bands,
        "rejection_reasons": rejection_reasons,
        "rejection_diagnostics": rejection_diagnostics,
        "last_candidate_table": last_candidate_table,
        "final_residuals": final_resid_df,
    }


def make_candidate_summary(candidate_table):
    if candidate_table is None or len(candidate_table) == 0:
        return ""

    rows = []

    for _, r in candidate_table.iterrows():
        rows.append(
            f"{r['band']}: score={r['score']:.2f}, "
            f"LOO_sig={r['residual_sigma_leave_one_out']:.2f}, "
            f"dchi2nu={r['red_chi2_improvement']:.2f}, "
            f"localmag={r['local_delta_mag']:.2f}, "
            f"pass={r['passes_rejection_test']}"
        )

    return "; ".join(rows)


def make_variable_or_mismatch_flag(final_residuals, red_chi2):
    if final_residuals is None or len(final_residuals) == 0:
        return False, 0

    n_bad = int(np.sum(final_residuals["abs_residual_sigma"] >= variable_flag_abs_resid_sigma))

    flag = (
        n_bad >= variable_flag_min_bad_bands
        or (
            np.isfinite(red_chi2)
            and red_chi2 >= variable_flag_red_chi2
        )
    )

    return bool(flag), n_bad


# ------------------------------------------------------------
# MARCS MODEL CLOSEST TO SP_ACE
# ------------------------------------------------------------

def find_closest_marcs_to_sp_ace(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    """
    Find the MARCS model closest to the SP_Ace atmospheric parameters.
    This is independent of the photometric chi2 fit.
    """

    grid = grid_base.copy()

    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    grid["SP_distance"] = (
        ((grid["teff"] - teff_sp) / 100.0) ** 2
        + ((grid["logg"] - logg_sp) / 0.5) ** 2
        + ((grid["feh"] - feh_sp) / 0.25) ** 2
    )

    if np.isfinite(afe_sp):
        grid["SP_distance"] += ((grid["alpha"] - afe_sp) / 0.20) ** 2

    closest = grid.sort_values("SP_distance").iloc[0].copy()
    closest["SP_closest_used_geom"] = used_geom

    return closest


def fit_single_cached_model_to_sed(model_row, sed):
    """
    Scale one cached MARCS model to the observed SED and compute chi2.
    Used for the MARCS model closest to SP_Ace.
    """

    model_cols = sed["model_col"].tolist()

    obs = sed["nu_fnu"].to_numpy(dtype=float)
    err = sed["nu_fnu_err"].to_numpy(dtype=float)
    mod = model_row[model_cols].to_numpy(dtype=float)

    good = (
        np.isfinite(obs)
        & np.isfinite(err)
        & np.isfinite(mod)
        & (obs > 0)
        & (err > 0)
        & (mod > 0)
    )

    if np.sum(good) < 2:
        return np.nan, np.nan, np.nan, int(np.sum(good))

    y = obs[good]
    s = err[good]
    m = mod[good]

    w = 1.0 / s**2

    scale = np.sum(m * y * w) / np.sum(m**2 * w)

    chi2 = np.sum(((y - scale * m) / s) ** 2)

    nfit = int(np.sum(good))
    dof = nfit - 1

    red_chi2 = chi2 / dof if dof > 0 else np.nan

    return scale, chi2, red_chi2, nfit


# ------------------------------------------------------------
# PLOTTER
# ------------------------------------------------------------

def get_smoothed_model_curve(model_row, wave_A, model_curve_cache):
    """
    Read and smooth one MARCS model spectrum for plotting.
    Returns unscaled smoothed curve.
    """

    model_path = resolve_model_path(model_row["path"], model_row["filename"], MARCS_DIR)

    if model_path in model_curve_cache:
        return model_curve_cache[model_path]

    flux_lambda = read_numeric_file(model_path)

    n = min(len(wave_A), len(flux_lambda))

    lam_um, nu_fnu_surface = marcs_flux_to_nuFnu(
        wave_A[:n],
        flux_lambda[:n]
    )

    mrange = (lam_um >= 0.30) & (lam_um <= 1.20)

    lam_um = lam_um[mrange]
    nu_fnu_surface = nu_fnu_surface[mrange]

    lam_smooth, flux_smooth = bin_model_in_log_lambda(
        lam_um,
        nu_fnu_surface,
        n_bins=model_smoothing_bins
    )

    model_curve_cache[model_path] = (lam_smooth, flux_smooth)

    return lam_smooth, flux_smooth


def plot_best_model_for_fiber(
    fiberid,
    row,
    sed_initial,
    sed_used,
    rejected_bands,
    best_phot,
    closest_sp_model,
    closest_sp_scale,
    closest_sp_red_chi2,
    cmd_info,
    wave_A,
    model_curve_cache,
    output_png
):
    if skip_existing_png and os.path.exists(output_png):
        return output_png

    # Photometric best MARCS model
    lam_best, flux_best_unscaled = get_smoothed_model_curve(
        best_phot,
        wave_A,
        model_curve_cache
    )

    flux_best_scaled = best_phot["scale"] * flux_best_unscaled

    # MARCS model closest to SP_Ace parameters
    lam_sp, flux_sp_unscaled = get_smoothed_model_curve(
        closest_sp_model,
        wave_A,
        model_curve_cache
    )

    flux_sp_scaled = closest_sp_scale * flux_sp_unscaled

    # Axis limits include all original points, including rejected ones.
    y_all = sed_initial["nu_fnu"].to_numpy(dtype=float)
    yerr_all = sed_initial["nu_fnu_err"].to_numpy(dtype=float)

    y_low = y_all - yerr_all
    y_high = y_all + yerr_all

    positive_data = np.concatenate([
        y_low[np.isfinite(y_low) & (y_low > 0)],
        y_high[np.isfinite(y_high) & (y_high > 0)]
    ])

    positive_model = flux_best_scaled[np.isfinite(flux_best_scaled) & (flux_best_scaled > 0)]
    positive_sp = flux_sp_scaled[np.isfinite(flux_sp_scaled) & (flux_sp_scaled > 0)]

    ymin = np.nanmin(np.concatenate([positive_data, positive_model, positive_sp]))
    ymax = np.nanmax(np.concatenate([positive_data, positive_model, positive_sp]))

    log_ymin = np.log10(ymin)
    log_ymax = np.log10(ymax)
# ------------------------------------------------------------
# Asymmetric y-axis padding
# Leave extra vertical room above the spectrum so the legend
# does not cover the top photometry/model region.
# ------------------------------------------------------------

    yrange = log_ymax - log_ymin if log_ymax > log_ymin else 1.0

    ypad_low = 0.10 * yrange

# Larger top padding for internal legend.
# 0.45 dex means the top of the plot is about 2.8 times above the highest point.
    ypad_high = max(0.45, 0.22 * yrange)

    ylim_low = 10.0 ** (log_ymin - ypad_low)
    ylim_high = 10.0 ** (log_ymax + ypad_high)



    teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
    logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
    feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
    afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

    # Normal height; legend is inside the plot, not above it.
    fig, ax = plt.subplots(figsize=(8.8, 6.4))

    # Used photometry: filled black circles
    used = sed_initial[~sed_initial["band"].isin(rejected_bands)].copy()

    ax.errorbar(
        used["lambda_um"],
        used["nu_fnu"],
        yerr=used["nu_fnu_err"],
        fmt="o",
        markersize=4.2,
        capsize=2.5,
        elinewidth=0.9,
        markeredgewidth=0.6,
        markerfacecolor="black",
        markeredgecolor="black",
        linestyle="none",
        label=f"Fiber {fiberid} fitted photometry"
    )

    # Rejected photometry: open red squares
    rejected = sed_initial[sed_initial["band"].isin(rejected_bands)].copy()

    if len(rejected) > 0:
        ax.errorbar(
            rejected["lambda_um"],
            rejected["nu_fnu"],
            yerr=rejected["nu_fnu_err"],
            fmt="s",
            markersize=5.0,
            capsize=2.5,
            elinewidth=0.9,
            markeredgewidth=1.0,
            markerfacecolor="none",
            markeredgecolor="red",
            ecolor="red",
            linestyle="none",
            label="Rejected photometry"
        )

    for _, r in sed_initial.iterrows():
        label_color = "red" if r["band"] in rejected_bands else "black"
        ax.annotate(
            r["band"],
            (r["lambda_um"], r["nu_fnu"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
            color=label_color
        )

    # Best photometric MARCS model
    phot_label = (
        "Best photometric MARCS "
        f"T={best_phot['teff']:.0f} K, "
        f"logg={best_phot['logg']:.1f}, "
        f"[Fe/H]={best_phot['feh']:.2f}, "
        f"[α/Fe]={best_phot['alpha']:.2f}, "
        f"χ²ν={best_phot['red_chi2']:.2f}"
    )

    ax.plot(
        lam_best,
        flux_best_scaled,
        linewidth=1.1,
        alpha=0.9,
        color="tab:blue",
        label=phot_label
    )

    # MARCS model closest to SP_Ace
    spclose_label = (
        "MARCS closest to SP_Ace "
        f"T={closest_sp_model['teff']:.0f} K, "
        f"logg={closest_sp_model['logg']:.1f}, "
        f"[Fe/H]={closest_sp_model['feh']:.2f}, "
        f"[α/Fe]={closest_sp_model['alpha']:.2f}, "
        f"χ²ν={closest_sp_red_chi2:.2f}"
    )

    ax.plot(
        lam_sp,
        flux_sp_scaled,
        linewidth=0.95,
        alpha=0.9,
        color="tab:orange",
        label=spclose_label
    )

    CaT_lines_um = [0.849802, 0.854209, 0.866214]

    for line_um in CaT_lines_um:
        ax.axvline(line_um, linestyle=":", linewidth=0.8, color="gray", alpha=0.8)

    rejected_text = ",".join(rejected_bands) if len(rejected_bands) > 0 else "none"

    textstr = (
        f"CMD region: {cmd_info['cmd_region']}\n"
        f"SP_Ace: T={teff_sp:.0f} K, logg={logg_sp:.2f}, [Fe/H]={feh_sp:.2f}, [α/Fe]={afe_sp:.2f}\n"
        f"Phot best: T={best_phot['teff']:.0f} K, logg={best_phot['logg']:.1f}, [Fe/H]={best_phot['feh']:.2f}, [α/Fe]={best_phot['alpha']:.2f}\n"
        f"SP-nearest MARCS: T={closest_sp_model['teff']:.0f} K, logg={closest_sp_model['logg']:.1f}, [Fe/H]={closest_sp_model['feh']:.2f}, [α/Fe]={closest_sp_model['alpha']:.2f}\n"
        f"Rejected bands: {rejected_text}"
    )

    # Fixed information panel: lower-left
    ax.text(
        0.03,
        0.04,
        textstr,
        transform=ax.transAxes,
        fontsize=7.6,
        va="bottom",
        ha="left",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            edgecolor="black",
            alpha=0.75
        ),
        zorder=10
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(0.32, 1.10)
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_xlabel("Wavelength [micron]")
    ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_title(f"Fiber {fiberid}: photometric MARCS fit vs SP_Ace-nearest MARCS model")

    ax.grid(alpha=0.25, which="both")

    # Fixed legend panel INSIDE the plot, upper-left.
    # It stays on the plot and no longer collides with the title.
    legend = ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.01, 0.99),
        fontsize=7.0,
        framealpha=0.88,
        borderaxespad=0.0,
        handlelength=2.4,
        ncol=1
    )

    legend.set_zorder(20)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)

    return output_png


# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------

if not os.path.exists(cache_file):
    raise FileNotFoundError(
        "Cache file does not exist. Run the MARCS synthetic cache-building cell first:\n"
        f"{cache_file}"
    )

summary_file = find_existing_file(summary_file_candidates)

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)
cache = pd.read_csv(cache_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(summary, on="fiberid", how="left", suffixes=("", "_SPsummary"))

df = df[np.isfinite(df["fiberid"])].copy()
df["fiberid"] = df["fiberid"].astype(int)

df = df.drop_duplicates(subset=["fiberid"]).sort_values("fiberid").reset_index(drop=True)

if max_fibers_to_run is not None:
    df = df.iloc[:max_fibers_to_run].copy()

grid_base = cache.copy()

if abundance_filter is not None:
    grid_base = grid_base[grid_base["abund"] == abundance_filter].copy()

if geometry_filter is not None:
    grid_base = grid_base[grid_base["geom"] == geometry_filter].copy()

if len(grid_base) == 0:
    raise ValueError("Cached MARCS grid is empty after abundance/geometry filters.")

wave_file = find_wavelength_file(MARCS_DIR)
wave_A = read_numeric_file(wave_file)

print(f"Number of fibers to process: {len(df)}")
print(f"Number of cached MARCS models before per-fiber restrictions: {len(grid_base)}")
print(f"Automatic photometry outlier rejection: {enable_photometry_outlier_rejection}")
print(f"Outlier threshold: {phot_outlier_sigma} sigma")
print(f"Maximum rejected bands per star: {phot_outlier_max_reject}")
print(f"Master output table:\n{master_output_csv}")
print(f"PNG output directory:\n{output_dir}")


# ------------------------------------------------------------
# BATCH LOOP
# ------------------------------------------------------------

master_rows = []
model_curve_cache = {}

for idx, row in df.iterrows():

    fiberid = int(row["fiberid"])

    if (idx % 10) == 0:
        print(f"\nProcessing fiber {idx+1} / {len(df)}: fiberid={fiberid}", flush=True)

    output_png = os.path.join(output_dir, f"SED_Fiber_{fiberid:04d}_MARCS_phot_vs_SPnearest_LOOreject.png")

    failed_result = {
        "fiberid": fiberid,
        "status": "failed",
        "error": "",
        "png_file": output_png,
    }

    try:
        teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
        logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
        feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
        afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

        if not np.isfinite(teff_sp) or not np.isfinite(logg_sp) or not np.isfinite(feh_sp):
            raise ValueError(
                f"Missing SP_Ace parameters: Teff={teff_sp}, logg={logg_sp}, FeH={feh_sp}"
            )

        manual_exclude = list(exclude_bands_global)

        if fiberid in per_fiber_exclude_bands:
            manual_exclude += list(per_fiber_exclude_bands[fiberid])

        manual_exclude = sorted(list(set(manual_exclude)))

        sed_initial = build_dereddened_sed(row, exclude_bands=manual_exclude)

        if len(sed_initial) < 2:
            raise ValueError("Fewer than two usable photometric bands.")

        cmd_info = get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp)

        n_cache_before = len(grid_base)

        grid_restricted, used_geom = restrict_grid_for_fiber(
            grid_base,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            cmd_info
        )

        fallback_used = False

        if len(grid_restricted) == 0 and allow_fallback_wider_box:
            grid_restricted = restrict_grid_fallback(
                grid_base,
                teff_sp,
                logg_sp,
                feh_sp,
                afe_sp
            )
            used_geom = "fallback"
            fallback_used = True

        n_cache_after = len(grid_restricted)

        if n_cache_after == 0:
            raise ValueError("No cached MARCS models left after restricted CMD/SP_Ace cuts.")

        if (idx % 10) == 0:
            print(f"  restricted models: {n_cache_after} / {n_cache_before}", flush=True)

        fit_info = fit_with_photometry_outlier_rejection(
            grid_restricted,
            sed_initial
        )

        best = fit_info["best"]
        sed_used = fit_info["sed_used"]
        rejected_bands = fit_info["rejected_bands"]
        rejection_reasons = fit_info["rejection_reasons"]
        final_residuals = fit_info["final_residuals"]
        last_candidate_table = fit_info["last_candidate_table"]

        closest_sp_model = find_closest_marcs_to_sp_ace(
            grid_base,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp
        )

        closest_sp_scale, closest_sp_chi2, closest_sp_red_chi2, closest_sp_nfit = fit_single_cached_model_to_sed(
            closest_sp_model,
            sed_used
        )

        possible_variable_or_mismatch, n_bad_bands_after_rejection = make_variable_or_mismatch_flag(
            final_residuals,
            best["red_chi2"]
        )

        if make_plots:
            png_file = plot_best_model_for_fiber(
                fiberid,
                row,
                sed_initial,
                sed_used,
                rejected_bands,
                best,
                closest_sp_model,
                closest_sp_scale,
                closest_sp_red_chi2,
                cmd_info,
                wave_A,
                model_curve_cache,
                output_png
            )
        else:
            png_file = ""

        initial_bands = ",".join(sed_initial["band"].tolist())
        used_bands = ",".join(sed_used["band"].tolist())
        rejected_bands_str = ",".join(rejected_bands)
        rejection_reasons_str = "; ".join(rejection_reasons)
        loo_candidate_summary = make_candidate_summary(last_candidate_table)

        if len(final_residuals) > 0:
            max_resid_band = final_residuals.iloc[0]["band"]
            max_abs_resid_sigma = final_residuals.iloc[0]["abs_residual_sigma"]
            max_signed_resid_sigma = final_residuals.iloc[0]["residual_sigma"]

            residual_summary = "; ".join([
                f"{r['band']}:{r['residual_sigma']:.2f}"
                for _, r in final_residuals.iterrows()
            ])
        else:
            max_resid_band = ""
            max_abs_resid_sigma = np.nan
            max_signed_resid_sigma = np.nan
            residual_summary = ""

        result_row = {
            "fiberid": fiberid,
            "status": "modeled",
            "error": "",
            "png_file": png_file,

            "RV": row["RV"] if "RV" in row.index else np.nan,
            "e_RV": row["e_RV"] if "e_RV" in row.index else np.nan,
            "S/N": row["S/N"] if "S/N" in row.index else np.nan,

            "u0": cmd_info["u0"],
            "i0": cmd_info["i0"],
            "ui0": cmd_info["ui0"],
            "cmd_region": cmd_info["cmd_region"],

            "initial_bands": initial_bands,
            "used_bands": used_bands,
            "manual_excluded_bands": ",".join(manual_exclude),
            "auto_rejected_bands": rejected_bands_str,
            "auto_rejection_reasons": rejection_reasons_str,
            "LOO_candidate_summary": loo_candidate_summary,

            "n_bands_initial": len(sed_initial),
            "n_bands_used": len(sed_used),

            "max_resid_band_after_rejection": max_resid_band,
            "max_abs_resid_sigma_after_rejection": max_abs_resid_sigma,
            "max_signed_resid_sigma_after_rejection": max_signed_resid_sigma,
            "band_residuals_after_rejection": residual_summary,

            "possible_variable_or_photometric_mismatch": possible_variable_or_mismatch,
            "n_bad_bands_after_rejection": n_bad_bands_after_rejection,

            "Teff_SP": teff_sp,
            "logg_SP": logg_sp,
            "FeH_SP": feh_sp,
            "aFe_SP": afe_sp,

            "best_Teff": best["teff"],
            "best_logg": best["logg"],
            "best_FeH": best["feh"],
            "best_aFe": best["alpha"],
            "best_geom": best["geom"],
            "best_abund": best["abund"],
            "best_vturb": best["vturb"],

            "best_scale": best["scale"],
            "chi2": best["chi2"],
            "red_chi2": best["red_chi2"],
            "nfit": best["nfit"],

            "best_filename": best["filename"],
            "best_path": best["path"],

            "SPnearest_Teff": closest_sp_model["teff"],
            "SPnearest_logg": closest_sp_model["logg"],
            "SPnearest_FeH": closest_sp_model["feh"],
            "SPnearest_aFe": closest_sp_model["alpha"],
            "SPnearest_geom": closest_sp_model["geom"],
            "SPnearest_abund": closest_sp_model["abund"],
            "SPnearest_vturb": closest_sp_model["vturb"],
            "SPnearest_scale": closest_sp_scale,
            "SPnearest_chi2": closest_sp_chi2,
            "SPnearest_red_chi2": closest_sp_red_chi2,
            "SPnearest_nfit": closest_sp_nfit,
            "SPnearest_distance": closest_sp_model["SP_distance"],
            "SPnearest_filename": closest_sp_model["filename"],
            "SPnearest_path": closest_sp_model["path"],

            "dTeff_phot_model_minus_SP": best["teff"] - teff_sp,
            "dlogg_phot_model_minus_SP": best["logg"] - logg_sp,
            "dFeH_phot_model_minus_SP": best["feh"] - feh_sp,
            "daFe_phot_model_minus_SP": best["alpha"] - afe_sp,

            "dTeff_SPnearest_minus_SP": closest_sp_model["teff"] - teff_sp,
            "dlogg_SPnearest_minus_SP": closest_sp_model["logg"] - logg_sp,
            "dFeH_SPnearest_minus_SP": closest_sp_model["feh"] - feh_sp,
            "daFe_SPnearest_minus_SP": closest_sp_model["alpha"] - afe_sp,

            "n_cache_before_priors": n_cache_before,
            "n_cache_after_priors": n_cache_after,
            "used_geom_preference": used_geom,
            "fallback_used": fallback_used,

            "teff_low": cmd_info["teff_low"],
            "teff_high": cmd_info["teff_high"],
            "logg_low": cmd_info["logg_low"],
            "logg_high": cmd_info["logg_high"],
            "feh_low": cmd_info["feh_low"],
            "feh_high": cmd_info["feh_high"],
            "alpha_low": cmd_info["alpha_low"],
            "alpha_high": cmd_info["alpha_high"],

            "teff_min_cmd": cmd_info["teff_min_cmd"],
            "teff_max_cmd": cmd_info["teff_max_cmd"],
            "logg_min_cmd": cmd_info["logg_min_cmd"],
            "logg_max_cmd": cmd_info["logg_max_cmd"],
            "teff_half_width": cmd_info["teff_half_width"],
            "logg_half_width": cmd_info["logg_half_width"],
            "feh_half_width": cmd_info["feh_half_width"],
            "alpha_half_width": cmd_info["alpha_half_width"],
        }

        master_rows.append(result_row)

    except Exception as e:
        failed_result["error"] = str(e)

        for col in [
            "RV", "e_RV", "S/N",
            "Teff_avg_SP", "logg_avg_SP", "FeH_avg_SP", "aFe_avg_SP",
            "umag", "Au", "imag", "Ai"
        ]:
            if col in row.index:
                failed_result[col] = row[col]

        master_rows.append(failed_result)


# ------------------------------------------------------------
# SAVE MASTER SUMMARY TABLE
# ------------------------------------------------------------

master = pd.DataFrame(master_rows)

master.to_csv(master_output_csv, index=False)

print("\nBatch restricted cached MARCS fitting complete.")
print(f"Modeled successfully: {(master['status'] == 'modeled').sum()}")
print(f"Failed/skipped:       {(master['status'] != 'modeled').sum()}")

print("\nMaster summary written to:")
print(master_output_csv)

print("\nPNG directory:")
print(output_dir)

print("\nFirst few rows of master table:")
print(master.head())

In [ ]:
# ============================================================
# STAGE 4: SPACE MODEL PLOTS + INTEGRATED FIBER DIAGNOSTICS
# ============================================================
# Makes plots like the fiber 215 comparison: top spectral truth model and residual,
# bottom the SEDfit_v2 MARCS plot.

import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec


def read_space_model(path):
    arr = np.loadtxt(path)
    if arr.ndim != 2 or arr.shape[1] < 4:
        raise ValueError(f"Unexpected SPACE model file: {path}")
    cols = ["wave_A", "obs_flux", "col2", "model_flux"]
    if arr.shape[1] >= 6:
        cols += ["col4", "weight"]
    if arr.shape[1] >= 7:
        cols += ["flag"]
    while len(cols) < arr.shape[1]:
        cols.append(f"col{len(cols)}")
    return pd.DataFrame(arr, columns=cols[:arr.shape[1]])


def robust_sky_mask(resid, fitmask=None, sigma=6.0):
    r = np.asarray(resid, dtype=float)
    good = np.isfinite(r)
    if fitmask is not None:
        good &= fitmask
    if good.sum() < 10:
        return good
    med = np.nanmedian(r[good])
    mad = np.nanmedian(np.abs(r[good] - med))
    scale = 1.4826 * mad if mad > 0 else np.nanstd(r[good])
    if not np.isfinite(scale) or scale <= 0:
        return good
    return good & (np.abs(r - med) < sigma * scale)


def find_model_file(fiberid, mode="alpha_metals"):
    fid = int(fiberid)
    search_dirs = [SPACE_OUTPUT_DIR, BASE_DIR, BASE_DIR / "space_models", OUT_DIR]
    if mode == "alpha_metals":
        patterns = [f"**/space_model_alpha_metals_{fid}_best_conservative_no_norm*.dat", f"**/space_model_alpha_metals_{fid}_*.dat"]
    else:
        patterns = [f"**/space_model_{fid}_best_conservative_no_norm*.dat", f"**/space_model_{fid}_*.dat"]
    for d in search_dirs:
        if not Path(d).exists():
            continue
        for pat in patterns:
            matches = sorted(Path(d).glob(pat))
            if matches:
                return matches[0]
    return None


def find_sed_plot(fiberid):
    fid = int(fiberid)
    search_dirs = [SED_OUTPUT_DIR, BASE_DIR / "MARCS_SED_PLOTS_RESTRICTED_LOO_REJECT_SPclosest_MARCS", OUT_DIR]
    pats = [f"*Fiber_{fid:04d}*.png", f"*Fiber_{fid:03d}*.png", f"*Fiber_{fid}*.png"]
    for d in search_dirs:
        if not Path(d).exists():
            continue
        for pat in pats:
            matches = sorted(Path(d).glob(pat))
            if matches:
                return matches[0]
    return None


def plot_spectral_truth_panel(model_path, fig, gs_top, title):
    df = read_space_model(model_path)
    wave = df["wave_A"].values
    obs = df["obs_flux"].values
    mod = df["model_flux"].values
    resid = obs - mod
    fitmask = df["weight"].values > 0 if "weight" in df.columns else np.ones(len(df), dtype=bool)
    skyok = robust_sky_mask(resid, fitmask=fitmask, sigma=6.0)

    ax = fig.add_subplot(gs_top[0])
    axr = fig.add_subplot(gs_top[1], sharex=ax)

    ax.plot(wave, obs, lw=0.6, label="Observed normalized spectrum")
    ax.plot(wave, mod, lw=1.0, label="Fortran SPACE model")
    ax.scatter(wave[fitmask], mod[fitmask], s=3, alpha=0.25, label="SPACE-weighted pixels")
    ax.scatter(wave[fitmask & ~skyok], obs[fitmask & ~skyok], s=10, marker="x", label="sky/outlier masked")
    for lo, hi in SPACE_WINDOWS:
        ax.axvspan(lo, hi, alpha=0.10)
    ax.set_ylim(0, 1.45)
    ax.set_ylabel("Normalized flux")
    ax.set_title(title)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8, loc="lower left")

    axr.plot(wave, resid, lw=0.55)
    axr.scatter(wave[fitmask & ~skyok], resid[fitmask & ~skyok], s=10, marker="x")
    axr.axhline(0, lw=0.8)
    axr.set_ylim(-0.28, 0.28)
    axr.set_ylabel("Obs. - model")
    axr.set_xlabel("Rest-frame wavelength [A]")
    axr.grid(alpha=0.2)

    absr = np.abs(resid[fitmask & skyok])
    qc = {
        "model_file": str(model_path),
        "n_fit_pixels": int(fitmask.sum()),
        "n_scored_pixels": int((fitmask & skyok).sum()),
        "sky_clipped_rms_resid": float(np.sqrt(np.nanmean(resid[fitmask & skyok]**2))) if (fitmask & skyok).sum() else np.nan,
        "sky_clipped_p95_abs_resid": float(np.nanpercentile(absr, 95)) if len(absr) else np.nan,
    }
    return qc


def make_integrated_fiber_plot(fiberid, prefer_mode="alpha_metals", show=False):
    fid = int(fiberid)
    model_path = find_model_file(fid, prefer_mode)
    mode_used = prefer_mode
    if model_path is None and prefer_mode == "alpha_metals":
        model_path = find_model_file(fid, "standard")
        mode_used = "standard"
    sed_path = find_sed_plot(fid)

    if model_path is None or sed_path is None:
        return {"fiberid": fid, "status": "missing_model_or_sed", "model_file": str(model_path) if model_path else "", "sed_plot": str(sed_path) if sed_path else ""}

    fig = plt.figure(figsize=(11, 14))
    outer = GridSpec(3, 1, height_ratios=[2.5, 0.75, 4.0], hspace=0.22, figure=fig)
    spec_gs = outer[:2].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
    qc = plot_spectral_truth_panel(model_path, fig, [spec_gs[0], spec_gs[1]], f"Fiber {fid}: Fortran {mode_used} spectral truth model")

    ax_sed = fig.add_subplot(outer[2])
    img = mpimg.imread(sed_path)
    ax_sed.imshow(img)
    ax_sed.set_axis_off()
    ax_sed.set_title(f"Fiber {fid}: photometric MARCS fit vs SP_Ace-nearest MARCS model", fontsize=12)

    fig.suptitle(f"Fiber {fid}: spectral truth fit + SED/SP-nearest MARCS check", fontsize=11, y=0.995)
    out_png = INTEGRATED_PLOT_DIR / f"fiber_{fid}_spectral_truth_plus_SED.png"
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    qc.update({"fiberid": fid, "status": "ok", "mode_used": mode_used, "model_file": str(model_path), "sed_plot": str(sed_path), "integrated_plot": str(out_png)})
    return qc

# Quick test: fiber 215
qc_215 = make_integrated_fiber_plot(215, show=True)
qc_215

In [ ]:
# ============================================================
# STAGE 5: MAKE INTEGRATED PLOTS FOR ALL VALID FIBERS
# ============================================================

if MAKE_INTEGRATED_PLOTS:
    # Candidate fiber list: any fiber with an alpha or standard model and an SED plot.
    model_fids = set()
    for d in [SPACE_OUTPUT_DIR, BASE_DIR, BASE_DIR / "space_models", OUT_DIR]:
        if Path(d).exists():
            for p in Path(d).glob("**/space_model*.dat"):
                fid = parse_fiber_id_from_name(p)
                if fid is not None:
                    model_fids.add(fid)
    sed_fids = set()
    for d in [SED_OUTPUT_DIR, BASE_DIR / "MARCS_SED_PLOTS_RESTRICTED_LOO_REJECT_SPclosest_MARCS", OUT_DIR]:
        if Path(d).exists():
            for p in Path(d).glob("*.png"):
                fid = parse_fiber_id_from_name(p)
                if fid is not None:
                    sed_fids.add(fid)
    fiber_list = sorted(model_fids & sed_fids)
    if MAX_FIBERS is not None:
        fiber_list = fiber_list[:MAX_FIBERS]
    print(f"Making integrated plots for {len(fiber_list)} fibers with both model and SED plot.")

    rows = []
    for i, fid in enumerate(fiber_list, start=1):
        if i == 1 or i % 25 == 0 or i == len(fiber_list):
            print(f"  {i}/{len(fiber_list)} fiber {fid}")
        rows.append(make_integrated_fiber_plot(fid, show=False))
    diagnostic_manifest = pd.DataFrame(rows)
else:
    diagnostic_manifest = pd.DataFrame()

manifest_path = TABLE_DIR / "integrated_diagnostic_plot_manifest.csv"
diagnostic_manifest.to_csv(manifest_path, index=False)
print("Wrote", manifest_path)
diagnostic_manifest.head()

In [ ]:
# ============================================================
# STAGE 6: FINAL BEST-PRACTICES SUMMARY TABLE
# ============================================================
# This merges the CaT/DP table, SP_Ace summary, SED summary, existing best-averages
# table if present, and the integrated diagnostic plot paths.


def read_optional(path):
    path = Path(path)
    return read_table_auto(path) if path.exists() else pd.DataFrame()

cat_dp = read_optional(TABLE_DIR / "CaT_master_with_Dias_Parisi_FeHDP.csv")
spavg = read_optional(SP_ACE_AVG_FILE)
sed_summary = read_optional(SED_SUMMARY_CSV)
best = read_optional(BEST_AVERAGES_FILE)
diag = read_optional(TABLE_DIR / "integrated_diagnostic_plot_manifest.csv")

# Start with the most complete table available.
if len(best) > 0 and "fiberid" in best.columns:
    master = best.copy()
elif len(cat_dp) > 0 and "fiberid" in cat_dp.columns:
    master = cat_dp.copy()
elif len(spavg) > 0 and "fiberid" in spavg.columns:
    master = spavg.copy()
else:
    raise ValueError("No table with fiberid found. Run earlier stages first.")

# Merge CaT/DP quantities if they are not already present.
if len(cat_dp) > 0 and "fiberid" in cat_dp.columns:
    add_cols = [c for c in cat_dp.columns if c not in master.columns or c in ["fiberid"]]
    master = master.merge(cat_dp[add_cols], on="fiberid", how="left", suffixes=("", "_cat"))

# Merge SP_Ace average columns with SPavg suffix to avoid overwriting adopted values.
if len(spavg) > 0 and "fiberid" in spavg.columns:
    sp_ren = spavg.copy()
    sp_ren = sp_ren.rename(columns={c: f"{c}_SPavg" for c in sp_ren.columns if c != "fiberid" and c in master.columns})
    master = master.merge(sp_ren, on="fiberid", how="left")

# Merge SED summary.
if len(sed_summary) > 0 and "fiberid" in sed_summary.columns:
    sed_ren = sed_summary.copy()
    sed_ren = sed_ren.rename(columns={c: f"SED_{c}" for c in sed_ren.columns if c != "fiberid" and c in master.columns})
    master = master.merge(sed_ren, on="fiberid", how="left")

# Merge diagnostic plot paths.
if len(diag) > 0 and "fiberid" in diag.columns:
    master = master.merge(diag[[c for c in diag.columns if c in ["fiberid", "integrated_plot", "model_file", "sed_plot", "status", "mode_used"]]], on="fiberid", how="left")

# Adopted reporting columns. Prefer existing best-rule columns when they exist.
def first_existing(row, cols):
    for c in cols:
        if c in row.index and pd.notna(row[c]):
            return row[c]
    return np.nan

master["RV_barycentric_report"] = master.apply(lambda r: first_existing(r, ["RV_barycentric", "RV_bary_CaT", "RV", "RV_barycentric_cat"]), axis=1)
master["e_RV_report"] = master.apply(lambda r: first_existing(r, ["e_RV_barycentric", "e_RV", "e_RVraw_CaT"]), axis=1)
master["Teff_report"] = master.apply(lambda r: first_existing(r, ["Teff_adopted", "Teff_SP", "Teff", "Teff_SPavg", "Teff_MARCS_SED"]), axis=1)
master["e_Teff_report"] = master.apply(lambda r: first_existing(r, ["e_Teff_adopted", "e_Teff_SP", "T_h", "T_h_SPavg"]), axis=1)
master["logg_report"] = master.apply(lambda r: first_existing(r, ["logg_adopted", "logg_SP", "logg", "logg_SPavg", "logg_MARCS_SED"]), axis=1)
master["e_logg_report"] = master.apply(lambda r: first_existing(r, ["e_logg_adopted", "e_logg_SP"]), axis=1)
master["FeH_report"] = master.apply(lambda r: first_existing(r, ["FeH_adopted", "FeH_SP", "FeH", "FeH_SPavg", "FeH_CaT_DP"]), axis=1)
master["e_FeH_report"] = master.apply(lambda r: first_existing(r, ["e_FeH_adopted", "e_FeH_SP", "e_FeH_CaT_DP", "e_FeH_CaT_DP_beta_only"]), axis=1)
master["aFe_report"] = master.apply(lambda r: first_existing(r, ["aFe_best_rule", "aFe_SP", "alphaFe", "alphaFe_SPavg"]), axis=1)
master["e_aFe_report"] = master.apply(lambda r: first_existing(r, ["e_aFe_best_rule", "e_aFe_SP"]), axis=1)

# Reporting flags
if "FeH_SP" in master.columns and "FeH_CaT_DP" in master.columns:
    master["FeH_SP_vs_DP_delta"] = pd.to_numeric(master["FeH_SP"], errors="coerce") - pd.to_numeric(master["FeH_CaT_DP"], errors="coerce")
    master["FeH_DP_caution"] = master["FeH_SP_vs_DP_delta"].abs() > 0.30
else:
    master["FeH_SP_vs_DP_delta"] = np.nan
    master["FeH_DP_caution"] = False

# SED caution is intentionally not an automatic rejection; variables and non-simultaneous photometry matter.
sed_cols = [c for c in master.columns if "variable" in c.lower() or "rejected" in c.lower() or "red_chi2" in c.lower()]
master["SED_photometric_caution"] = False
for c in sed_cols:
    vals = master[c]
    if vals.dtype == object:
        master["SED_photometric_caution"] |= vals.astype(str).str.contains("True|variable|reject|bad|caution", case=False, na=False)
    else:
        if "chi2" in c.lower():
            master["SED_photometric_caution"] |= pd.to_numeric(vals, errors="coerce") > 5

master["recommended_reporting_note"] = "Use SP_Ace/SPACE spectral values if model is visually acceptable; use CaT-DP as metallicity check/fallback; use SED as photometric plausibility and variability caution."
master.loc[master["FeH_DP_caution"], "recommended_reporting_note"] += " FeH_SP and FeH_CaT_DP differ by >0.30 dex."
master.loc[master["SED_photometric_caution"], "recommended_reporting_note"] += " SED/photometry caution: possible variability, rejected band, or non-simultaneous photometry."

full_out = TABLE_DIR / "NGC6569_full_integrated_best_practices_master_table.csv"
report_cols = [
    "fiberid", "RV_barycentric_report", "e_RV_report", "Teff_report", "e_Teff_report",
    "logg_report", "e_logg_report", "FeH_report", "e_FeH_report",
    "aFe_report", "e_aFe_report", "FeH_CaT_DP", "FeH_SP_vs_DP_delta",
    "FeH_DP_caution", "SED_photometric_caution", "integrated_plot", "recommended_reporting_note"
]
report_cols = [c for c in report_cols if c in master.columns]
compact_out = TABLE_DIR / "NGC6569_publication_best_practices_reporting_table.csv"
master.to_csv(full_out, index=False)
master[report_cols].to_csv(compact_out, index=False)

print("Wrote full master table:", full_out)
print("Wrote compact reporting table:", compact_out)
master[report_cols].head()

In [ ]:
# ============================================================
# DISPLAY THE COMPLETE COMPARISON FOR ONE FIBER
# ============================================================
# Change fiber_to_show to any fiber with a valid model + SED fit.

from IPython.display import Image, display

fiber_to_show = 215
png = INTEGRATED_PLOT_DIR / f"fiber_{fiber_to_show}_spectral_truth_plus_SED.png"
print(png)
print("Exists?", png.exists())
if png.exists():
    display(Image(filename=str(png)))
else:
    print("Plot does not exist yet. Run Stage 5 first.")